# SkillGuard — edge compliance pipeline

This notebook implements the production-shaped video path: **RF-DETR → ByteTrack → classroom zone → camera-quality checks → temporal compliance rules → evidence package**.

It is designed for testing and validation in Colab. Production should run the same logic on a centre-side edge computer or approved server, not continuously in Colab.

## Safety and interpretation
- This is **aggregate occupancy verification**, not facial recognition or individual attendance.
- It must never treat a detector count as ground-truth attendance.
- When crowding/occlusion or camera quality prevents a defensible conclusion, it outputs `UNVERIFIABLE` rather than an attendance violation.
- Upload only footage you are authorised to process.


## Colab runtime

Choose **Runtime → Change runtime type → Hardware accelerator: GPU**. A T4 GPU is sufficient for the initial Medium-model experiment. Run cells in order.


In [ ]:
# 1. Confirm a GPU is available
!nvidia-smi

import sys
import torch

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Select a GPU runtime, restart the runtime, and rerun this cell.')


In [ ]:
# 2. Install pinned runtime dependencies
!pip install -q -U rfdetr supervision trackers opencv-python-headless pandas tqdm

import cv2
import numpy as np
import pandas as pd
import supervision as sv
from trackers import ByteTrackTracker

print('OpenCV:', cv2.__version__)
print('Supervision:', sv.__version__)


In [ ]:
# 3. Upload one authorised MP4/MOV/AVI video
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No video selected. Rerun this cell and upload one video.')

VIDEO_INPUT = next(iter(uploaded))
VIDEO_OUTPUT = 'skillguard_annotated.mp4'
print('Video input:', VIDEO_INPUT)


In [ ]:
# 4. Inspect the source video and set the compliance configuration
video_info = sv.VideoInfo.from_video_path(VIDEO_INPUT)
if video_info.total_frames <= 0:
    raise RuntimeError('Could not read the video. Re-encode/upload a standard H.264 MP4.')

FRAME_WIDTH, FRAME_HEIGHT = video_info.resolution_wh
print(video_info)

# EDIT THESE VALUES FOR EACH BATCH / CAMERA CONFIGURATION.
CENTRE_ID = 'TC-DEMO-001'
CAMERA_ID = 'CAM-CLASSROOM-A'
BATCH_ID = 'BATCH-DEMO-001'
REPORTED_ATTENDANCE = 13  # Enter the officially reported batch attendance.
CONFIDENCE_THRESHOLD = 0.35
NMS_IOU_THRESHOLD = 0.70
WINDOW_SECONDS = 10       # Use 5-15 min in a real deployment; 10 sec makes a short demo observable.
ATTENDANCE_TOLERANCE = 0.20  # Alert only when count is 20%+ below reported attendance.
MIN_WINDOW_CONFIDENCE = 0.70
MIN_WINDOW_STABILITY = 0.65

# IMPORTANT: High crowding means detector undercount must be treated as uncertain, not a violation.
# Choose one: 'low', 'medium', 'high'. For your tightly seated 13-student footage, keep 'high'.
OCCLUSION_RISK = 'high'

# Classroom-zone polygon. It initially covers the full frame.
# Replace these points after inspecting the frame below so corridors/doors are excluded.
ZONE_POLYGON = np.array([
    [0, 0],
    [FRAME_WIDTH - 1, 0],
    [FRAME_WIDTH - 1, FRAME_HEIGHT - 1],
    [0, FRAME_HEIGHT - 1],
], dtype=np.int32)

print('Video resolution:', FRAME_WIDTH, 'x', FRAME_HEIGHT)
print('Reported attendance:', REPORTED_ATTENDANCE)
print('Occlusion risk:', OCCLUSION_RISK)


In [ ]:
# 5. Inspect the first frame and classroom zone.
# If doors/corridors appear, edit ZONE_POLYGON in the previous cell and rerun this cell.
from matplotlib import pyplot as plt

capture = cv2.VideoCapture(VIDEO_INPUT)
ok, first_frame = capture.read()
capture.release()
if not ok:
    raise RuntimeError('Unable to retrieve the first video frame.')

zone_preview = first_frame.copy()
cv2.polylines(zone_preview, [ZONE_POLYGON], isClosed=True, color=(0, 255, 255), thickness=3)
plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(zone_preview, cv2.COLOR_BGR2RGB))
plt.title('Yellow outline = classroom/workshop counting zone')
plt.axis('off');


In [ ]:
# 6. Load RF-DETR and initialise ByteTrack
from rfdetr import RFDETRMedium
from rfdetr.assets.coco_classes import COCO_CLASSES

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PERSON_CLASS_ID = COCO_CLASSES.index('person')

print(f'Loading RF-DETR Medium on {DEVICE}...')
model = RFDETRMedium(device=DEVICE)
tracker = ByteTrackTracker(
    frame_rate=video_info.fps,
    track_activation_threshold=CONFIDENCE_THRESHOLD,
)
zone = sv.PolygonZone(
    polygon=ZONE_POLYGON,
    triggering_anchors=(sv.Position.BOTTOM_CENTER,),
)
print('Model and tracker are ready.')


In [ ]:
# 7. Production-shaped pipeline helpers: camera quality, zone counting, tracking, and annotation
from collections import Counter
from tqdm.auto import tqdm

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

def camera_quality(frame_bgr):
    """Returns a conservative visual quality score and evidence labels."""
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    brightness = float(gray.mean())
    blur_variance = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    flags = []
    brightness_score = min(1.0, max(0.0, brightness / 70.0))
    blur_score = min(1.0, max(0.0, blur_variance / 90.0))

    if brightness < 35:
        flags.append('dark')
    if blur_variance < 35:
        flags.append('blurred_or_obstructed')

    score = round(0.55 * brightness_score + 0.45 * blur_score, 3)
    status = 'healthy' if score >= 0.70 else ('degraded' if score >= 0.45 else 'unhealthy')
    return score, status, flags, round(brightness, 1), round(blur_variance, 1)

def annotate(frame_bgr, tracked_zone_detections, count, quality_status):
    labels = []
    for confidence, tracker_id in zip(
        tracked_zone_detections.confidence, tracked_zone_detections.tracker_id
    ):
        track_label = 'pending' if tracker_id == -1 else str(int(tracker_id))
        labels.append(f'person #{track_label} {confidence:.2f}')

    output = box_annotator.annotate(scene=frame_bgr.copy(), detections=tracked_zone_detections)
    output = label_annotator.annotate(scene=output, detections=tracked_zone_detections, labels=labels)
    cv2.polylines(output, [ZONE_POLYGON], isClosed=True, color=(0, 255, 255), thickness=2)
    cv2.rectangle(output, (8, 8), (535, 94), (18, 26, 44), -1)
    cv2.putText(output, f'Visible tracked people: {count}', (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(output, f'Camera quality: {quality_status}', (20, 72), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return output


In [ ]:
# 8. Run RF-DETR + ByteTrack + zone filtering and write the annotated output video
# This may take several minutes on longer files.
second_samples = []
seen_track_ids = set()
sample_every_n_frames = max(1, round(video_info.fps))

frames = sv.get_video_frames_generator(source_path=VIDEO_INPUT)
with sv.VideoSink(target_path=VIDEO_OUTPUT, video_info=video_info) as sink:
    for frame_index, frame_bgr in enumerate(tqdm(frames, total=video_info.total_frames)):
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        detections = model.predict(frame_rgb, threshold=CONFIDENCE_THRESHOLD)
        detections = detections[detections.class_id == PERSON_CLASS_ID]
        detections = detections.with_nms(threshold=NMS_IOU_THRESHOLD)

        tracked = tracker.update(detections)
        tracked = tracked[tracked.tracker_id != -1]
        in_zone_mask = zone.trigger(tracked)
        in_zone = tracked[in_zone_mask]

        valid_ids = [int(track_id) for track_id in in_zone.tracker_id if track_id != -1]
        seen_track_ids.update(valid_ids)
        quality_score, quality_status, quality_flags, brightness, blur_variance = camera_quality(frame_bgr)

        if frame_index % sample_every_n_frames == 0:
            detection_confidence = float(np.mean(in_zone.confidence)) if len(in_zone) else 0.0
            second_samples.append({
                'time_seconds': round(frame_index / video_info.fps, 2),
                'visible_tracked_people': int(len(in_zone)),
                'mean_detection_confidence': round(detection_confidence, 3),
                'camera_quality_score': quality_score,
                'camera_quality_status': quality_status,
                'camera_quality_flags': ', '.join(quality_flags) if quality_flags else 'none',
                'brightness': brightness,
                'blur_variance': blur_variance,
                'active_track_ids': ','.join(map(str, valid_ids)),
            })

        sink.write_frame(frame=annotate(frame_bgr, in_zone, len(in_zone), quality_status))

print('Annotated video saved to:', VIDEO_OUTPUT)
print('Unique visible tracks observed:', len(seen_track_ids))


In [ ]:
# 9. Aggregate observations into time windows and apply conservative compliance rules
samples_df = pd.DataFrame(second_samples)
if samples_df.empty:
    raise RuntimeError('No samples were produced. Check the source video and model output.')

samples_df['window_id'] = (samples_df['time_seconds'] // WINDOW_SECONDS).astype(int)

windows_df = (
    samples_df.groupby('window_id')
    .agg(
        start_time_seconds=('time_seconds', 'min'),
        end_time_seconds=('time_seconds', 'max'),
        observed_median=('visible_tracked_people', 'median'),
        observed_mean=('visible_tracked_people', 'mean'),
        observed_min=('visible_tracked_people', 'min'),
        observed_max=('visible_tracked_people', 'max'),
        detection_confidence=('mean_detection_confidence', 'mean'),
        camera_quality_score=('camera_quality_score', 'mean'),
    )
    .reset_index()
)

windows_df['observed_median'] = windows_df['observed_median'].round(1)
windows_df['observed_mean'] = windows_df['observed_mean'].round(1)
windows_df['detection_confidence'] = windows_df['detection_confidence'].round(3)
windows_df['camera_quality_score'] = windows_df['camera_quality_score'].round(3)
windows_df['count_stability'] = (
    1 - ((windows_df['observed_max'] - windows_df['observed_min']) / windows_df['observed_median'].clip(lower=1))
).clip(lower=0, upper=1).round(3)

def classify_window(row):
    # Never create a punitive attendance result from a poor/occluded visual source.
    if row.camera_quality_score < MIN_WINDOW_CONFIDENCE:
        return 'CAMERA_UNHEALTHY'

    below_expected = row.observed_median < REPORTED_ATTENDANCE * (1 - ATTENDANCE_TOLERANCE)
    reliable_visual_evidence = (
        row.detection_confidence >= MIN_WINDOW_CONFIDENCE
        and row.count_stability >= MIN_WINDOW_STABILITY
        and OCCLUSION_RISK != 'high'
    )

    if below_expected and reliable_visual_evidence:
        return 'POTENTIAL_ATTENDANCE_DISCREPANCY'
    if below_expected:
        return 'UNVERIFIABLE'
    return 'COMPLIANT'

windows_df['reported_attendance'] = REPORTED_ATTENDANCE
windows_df['compliance_status'] = windows_df.apply(classify_window, axis=1)
windows_df['reason'] = windows_df.apply(
    lambda r: (
        'Observed count is below threshold, but crowding/occlusion or low stability makes the result non-defensible.'
        if r.compliance_status == 'UNVERIFIABLE' else
        'Camera quality is too low for occupancy verification.'
        if r.compliance_status == 'CAMERA_UNHEALTHY' else
        'Persistent observed occupancy is materially below reported attendance with adequate visual confidence.'
        if r.compliance_status == 'POTENTIAL_ATTENDANCE_DISCREPANCY' else
        'Observed occupancy is within configured tolerance.'
    ),
    axis=1,
)

display(windows_df)


In [ ]:
# 10. Build production-style event records and extract short evidence clips
import json
import os
from pathlib import Path
from datetime import datetime, timezone

EVIDENCE_DIR = Path('evidence')
EVIDENCE_DIR.mkdir(exist_ok=True)

def export_evidence_clip(source_path, start_seconds, end_seconds, target_path):
    cap = cv2.VideoCapture(source_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(target_path), fourcc, fps, (width, height))
    cap.set(cv2.CAP_PROP_POS_MSEC, max(0, start_seconds) * 1000)
    while cap.get(cv2.CAP_PROP_POS_MSEC) / 1000 <= end_seconds:
        ok, frame = cap.read()
        if not ok:
            break
        writer.write(frame)
    writer.release()
    cap.release()

events = []
for _, row in windows_df.iterrows():
    if row.compliance_status in {'COMPLIANT'}:
        continue

    event_id = f"EVT-{CENTRE_ID}-{int(row.window_id):04d}"
    clip_path = EVIDENCE_DIR / f'{event_id}.mp4'
    export_evidence_clip(
        VIDEO_INPUT,
        max(0, float(row.start_time_seconds) - 3),
        float(row.end_time_seconds) + 3,
        clip_path,
    )

    events.append({
        'event_id': event_id,
        'event_type': row.compliance_status.lower(),
        'centre_id': CENTRE_ID,
        'camera_id': CAMERA_ID,
        'batch_id': BATCH_ID,
        'window_start_seconds': float(row.start_time_seconds),
        'window_end_seconds': float(row.end_time_seconds),
        'reported_attendance': REPORTED_ATTENDANCE,
        'observed_median_visible_people': float(row.observed_median),
        'camera_quality_score': float(row.camera_quality_score),
        'detection_confidence': float(row.detection_confidence),
        'count_stability': float(row.count_stability),
        'occlusion_risk': OCCLUSION_RISK,
        'reason': row.reason,
        'evidence_clip': str(clip_path),
        'review_required': True,
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    })

print(json.dumps(events, indent=2))


In [ ]:
# 11. Save the edge-agent output package: samples, windows, events, and run configuration
samples_df.to_csv('skillguard_second_samples.csv', index=False)
windows_df.to_csv('skillguard_compliance_windows.csv', index=False)

run_package = {
    'run_metadata': {
        'centre_id': CENTRE_ID,
        'camera_id': CAMERA_ID,
        'batch_id': BATCH_ID,
        'reported_attendance': REPORTED_ATTENDANCE,
        'occlusion_risk': OCCLUSION_RISK,
        'model': 'RFDETRMedium',
        'tracker': 'ByteTrackTracker',
        'privacy_mode': 'aggregate occupancy; no facial identification',
    },
    'events': events,
}
with open('skillguard_event_package.json', 'w') as file:
    json.dump(run_package, file, indent=2)

print('Created:')
print('- skillguard_annotated.mp4')
print('- skillguard_second_samples.csv')
print('- skillguard_compliance_windows.csv')
print('- skillguard_event_package.json')
print('- evidence/*.mp4')


In [ ]:
# 12. Preview the annotated result
from IPython.display import Video, display
display(Video(VIDEO_OUTPUT, embed=True, width=900))


In [ ]:
# 13. Download the complete run package
!zip -qr skillguard_run_package.zip evidence skillguard_annotated.mp4 skillguard_second_samples.csv skillguard_compliance_windows.csv skillguard_event_package.json
from google.colab import files
files.download('skillguard_run_package.zip')


## Production hand-off

Use this notebook to validate the rules and prepare data. For a production edge agent:

1. Fine-tune RF-DETR on authorised, labelled footage from the actual camera angles.
2. Export the approved model to ONNX/TensorRT only after measurement on a held-out camera/site.
3. Run this pipeline continuously in Docker on an edge device, with SQLite for offline event queueing.
4. Send normal metadata periodically; send a short encrypted clip only for a review-required event.
5. Enforce role-based evidence access, retention policies, audit logs, and human review.

For the current 13-student, heavily occluded footage, keep `OCCLUSION_RISK = 'high'`. An undercount should become `UNVERIFIABLE`, not an automatic attendance discrepancy.
